In [40]:
import nptdms
import glob
import pandas as pd
import numpy as np
import datetime
import itertools
from scipy.signal import argrelextrema

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns # advanced vizs

# Set Color Palettes
palette1 = itertools.cycle(sns.color_palette(palette='Set1'))
palette2 = itertools.cycle(sns.color_palette(palette='Set2'))

In [41]:
# Define the path to your TDMS files
tdms_path = "D:\\TDMS\\Different Test Cases\\11.Right Two Crack at top & two crack at bottom & left two crack at top\\*.tdms"

# Get a list of file paths matching the pattern
file_list = glob.glob(tdms_path)

# Loop over the files and read them into pandas data frames
dfs = []
for file in file_list: # access file sotred in the file list
    with nptdms.TdmsFile.read(file) as tdms_file: #read tdms file in the folders
        groups = tdms_file.groups() #access groups in the tdms files (tdms_file.group() is the function for access tdms group)
        for group in groups: #iterate the group
            data = group.as_dataframe() #make individual file as dataframe
            dfs.append(data)        
# Concatenate the data frames into a single data frame
df = pd.concat(dfs, ignore_index=True) #Ignore index command is to remove index conflict while merge multiple files

In [42]:
# Change str format to datetime datatype
df['Time'] = pd.to_datetime((df.apply(lambda row: f"{row['Year']}-{row['Month']}-{row['Day']}-{row['Time(Hrs)']:02}:{row['Time(Min)']:02}:{row['Time(Sec)']:02}", axis=1)), format='%Y.0-%m.0-%d.0-%H.0:%M.0:%S.%f')

In [43]:
df.drop_duplicates(subset=['Time'],keep='first',inplace=True)
df.Time.duplicated().sum()

0

In [44]:
#Set Timeline column as index column
df.set_index('Time', inplace=True)

In [45]:
df.index.duplicated().sum()

0

In [46]:
df.columns

Index(['Day', 'Month', 'Year', 'Time(Hrs)', 'Time(Min)', 'Time(Sec)',
       'Cycle-detect', 'P1-CPM', 'P1-Cycle-Count', 'P1-Water Suction Pressure',
       'P1-Channel-1', 'P1-Water Discharge Pressure', 'P1-Channel-3',
       'P1-Air-Supply-pressure', 'P1-Water Suction Flowrate',
       'P1-Water Discharge Flowrate', 'P1-Air Supply Flowrate'],
      dtype='object')

# Drop trivial columns

In [47]:
df.drop(columns=['Day', 'Month', 'Year', 'Time(Hrs)', 'Time(Min)', 'Time(Sec)',
       'Cycle-detect', 'P1-CPM', 'P1-Cycle-Count', 'P1-Water Suction Pressure',
       'P1-Channel-1','P1-Channel-3',
       'P1-Air-Supply-pressure', 'P1-Water Suction Flowrate',
       'P1-Water Discharge Flowrate', 'P1-Air Supply Flowrate'], axis=1, inplace=True)

# Cal min&max for all measuring parameters

In [48]:
MinMaxdf = pd.DataFrame(index=df.index,)

for column in df.columns:
    MinMaxdf['min_{}'.format(column)] = np.nan
    MinMaxdf['max_{}'.format(column)] = np.nan
    MinMaxdf['min_{}'.format(column)] = df.loc[MinMaxdf['min_{}'.format(column)].index[argrelextrema(df[column].values, np.less_equal, order=1)[0]], column]
    MinMaxdf['max_{}'.format(column)] = df.loc[MinMaxdf['max_{}'.format(column)].index[argrelextrema(df[column].values, np.greater_equal, order=1)[0]], column]

In [49]:
MinMaxdf.head(50)

,min_P1-Water Discharge Pressure,max_P1-Water Discharge Pressure
Time,,
2023-08-28 12:45:40.216,0.000000,0.000000
2023-08-28 12:45:40.240,0.000000,0.000000
2023-08-28 12:45:40.248,0.000000,0.000000
2023-08-28 12:45:40.271,0.000000,NaN
2023-08-28 12:45:40.301,NaN,NaN
2023-08-28 12:45:40.331,NaN,14.150580
2023-08-28 12:45:40.360,14.010488,NaN
2023-08-28 12:45:40.390,NaN,NaN
2023-08-28 12:45:40.420,NaN,14.150580


In [ ]:
wdpmax_df = pd.DataFrame(data=MinMaxdf['max_P1-Water Discharge Pressure'])

In [ ]:
wdpmin_df = pd.DataFrame(data=MinMaxdf['min_P1-Water Discharge Pressure'])

In [ ]:
wdpmax_df.reset_index(inplace=True)

In [ ]:
wdpmin_df.reset_index(inplace=True)

In [ ]:
wdpmax_df.drop(columns=['Time'], axis=1, inplace=True)

In [ ]:
wdpmin_df.drop(columns=['Time'], axis=1, inplace=True)

In [ ]:
wdpmax_df.dropna(inplace=True)

In [ ]:
wdpmin_df.dropna(inplace=True)

In [ ]:
wdpmax_df.reset_index(inplace=True)

In [ ]:
wdpmin_df.reset_index(inplace=True)

In [ ]:
wdpmin_df.drop(columns=['index'], axis=1, inplace=True)

In [ ]:
wdpmin_df.head()

In [ ]:
df['type']=np.where(df.index%2==0, 'type_a', 'type_b') 

In [ ]:
if df.index%2==0:
    

# Create list of columns in MinMaxdf

In [ ]:
min_cols = [col for col in MinMaxdf.columns if 'min' in col]
max_cols = [col for col in MinMaxdf.columns if 'max' in col]

# Cal average and sum% of MinMaxdf 

In [ ]:
#Create Average and Sum% data frame
avg_df = pd.DataFrame(index=df.index)
sumpct_df = pd.DataFrame(index=df.index)

# Calculate average and sum% of MinMaxdf 
for column in MinMaxdf.columns:
    if column in min_cols:
        avg_df['avg_{}'.format(column)] = np.nan #Create nan column in avg_df
        sumpct_df['sumpct_{}'.format(column)] = np.nan #Create nan column in sumpct_df
        avg_df['avg_{}'.format(column)] = MinMaxdf.groupby(MinMaxdf.index)[column].mean() #cal groupby average of min valuse of measuring parameters 
        sumpct_df['sumpct_{}'.format(column)] = MinMaxdf.groupby(MinMaxdf.index)[column].sum(min_count=1).pct_change() #cal sum% of minimum valuse of measuring parameters
    elif column in max_cols:
        avg_df['avg_{}'.format(column)] = np.nan #Create nan column in avg_df
        sumpct_df['sumpct_{}'.format(column)] = np.nan #Create nan column in avg_df
        avg_df['avg_{}'.format(column)] = MinMaxdf.groupby(MinMaxdf.index)[column].mean() #cal groupby average of max valuse of measuring parameters
        sumpct_df['sumpct_{}'.format(column)] = MinMaxdf.groupby(MinMaxdf.index)[column].sum(min_count=1).pct_change() #cal sum% of max valuse of measuring parameters

# Data Visualization 
# Min&Max Values 

In [ ]:
# Year on Year box plots of min&max values of parameters

# Boxplots of the variables

a = len(MinMaxdf.columns)  # number of rows
b = 1  # number of columns
c = 1  # initialize plot counter

fig1 = plt.figure(figsize=(15,60))

for column in min_cols:
    plt.subplot(a, b, c)
    plt.title('Hour on Hour box plots for {}'.format(column), fontsize=14)
    sns.boxplot(x=MinMaxdf.index.hour, y=MinMaxdf[column], palette='Set1');
    c = c + 1

plt.tight_layout()
plt.show()

# Average Values of Min&Max

In [ ]:
# Year on Year box plots of average values of MinMaxdf

# Boxplots of the variables

a = len(avg_df.columns)  # number of rows
b = 2  # number of columns
c = 1  # initialize plot counter

fig1 = plt.figure(figsize=(15,60))

for column in avg_df.columns:
    plt.subplot(a, b, c)
    plt.title('Hour on Hour box plots for {}'.format(column), fontsize=14)
    sns.boxplot(x=avg_df.index.hour, y=avg_df[column], palette='Set1');
    c = c + 1

plt.tight_layout()
plt.show()

# Sum% Value of Min&Max

In [ ]:
# Year on Year box plots of sum% of MinMaxdf

# Boxplots of the variables

a = len(sumpct_df.columns)  # number of rows
b = 1  # number of columns
c = 1  # initialize plot counter

fig1 = plt.figure(figsize=(15,60))

for column in sumpct_df.columns:
    plt.subplot(a, b, c)
    plt.title("Sum Percent Change Over Time {}".format(column), fontsize=14)
    sns.lineplot(x=sumpct_df.index.hour, y=sumpct_df[column], marker='o');
    plt.grid()
    c = c + 1

plt.tight_layout()
plt.show()